# Cattle Re-ID — Etapa 3: mejoras del embedding (SupCon heavy-aug + DINOv2 fine-tune)

El diagnóstico de clustering mostró que el techo estaba en el **embedding**, no en el clusterer
(NMI 0.93 >> ARI 0.54 = clusters puros pero sobre-partidos; margen inter-vaca ~0.02 en coseno).
Este notebook entrena dos encoders nuevos para apretar la varianza intra-identidad y los compara
contra el SupCon original (0.542), ImageNet y DINOv2 congelado.

- **supcon_heavyaug** — ResNet-50 + SupCon + augmentation MUCHO más fuerte (foco en el hocico).
- **supcon_dinov2_heavyaug** — DINOv2 fine-tuneado *sin freeze* + SupCon + heavy aug.

Métrica primaria: **HDBSCAN ARI** con params fijos (número honesto). El sweep de epsilon es un
diagnóstico (oráculo, NO reportable). Robusto para correr desatendido: si un experimento falla,
sigue con el resto; checkpoints, logs y resultados se guardan a Drive.

## 1. Montar Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/cattle_reid')   # ajustá si tu carpeta se llama distinto
assert DRIVE_ROOT.is_dir(), f'No existe {DRIVE_ROOT}.'
print('contenido:', [p.name for p in DRIVE_ROOT.iterdir()])

Mounted at /content/drive
contenido: ['BeefCattle_Muzzle_Individualized.zip', 'outputs', 'dataset_caras_bovinos.zip', 'gradcam_test', 'CMPD300_Baseline.zip', 'Untitled0.ipynb', 'model_cache', 'Untitled']


## 2. Cachear modelos en Drive (ANTES de importar torch)

Setea `TORCH_HOME` (ResNet ImageNet) y `HF_HOME` (DINOv2) apuntando a Drive: se descargan una
vez y se reusan en cada sesión.

In [2]:
import os
CACHE = DRIVE_ROOT / 'model_cache'
(CACHE/'torch').mkdir(parents=True, exist_ok=True)
(CACHE/'hf').mkdir(parents=True, exist_ok=True)
os.environ['TORCH_HOME'] = str(CACHE/'torch')
os.environ['HF_HOME']    = str(CACHE/'hf')
print('TORCH_HOME=', os.environ['TORCH_HOME'])
print('HF_HOME   =', os.environ['HF_HOME'])

TORCH_HOME= /content/drive/MyDrive/cattle_reid/model_cache/torch
HF_HOME   = /content/drive/MyDrive/cattle_reid/model_cache/hf


## 3. GPU

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), 'Sin GPU: Entorno -> Cambiar tipo -> GPU (L4 recomendado).'
print('GPU:', torch.cuda.get_device_name(0))

## 4. Código + dependencias

In [3]:
import shutil
os.chdir('/content')
REPO_URL = 'https://github.com/benjaminvitale/tp-final-vision2-Pujol-Vitale.git'
BRANCH   = 'main'
REPO_DIR = '/content/tp-final-vision2-Pujol-Vitale'
if os.path.exists(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1
!pip -q install 'transformers==4.40.2'

Cloning into '/content/tp-final-vision2-Pujol-Vitale'...
remote: Enumerating objects: 361, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 361 (delta 23), reused 24 (delta 12), pack-reused 305 (from 1)
Receiving objects: 100% (361/361), 324.29 KiB | 17.07 MiB/s, done.
Resolving deltas: 100% (198/198), done.
/content/tp-final-vision2-Pujol-Vitale
44a1b55 (HEAD -> main, origin/main, origin/HEAD) Etapa 3: notebook nocturno — agregar ablación DINOv2 + strong-aug
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 110.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following 

## 5. Persistir outputs en Drive (checkpoints/logs/resultados no se pierden)

In [4]:
DRIVE_OUTPUTS = DRIVE_ROOT / 'outputs'
for sub in ('checkpoints', 'logs', 'results'):
    ds = DRIVE_OUTPUTS / sub; ds.mkdir(parents=True, exist_ok=True)
    ls = Path(REPO_DIR) / 'outputs' / sub
    if ls.exists() and not ls.is_symlink(): shutil.rmtree(ls)
    if not ls.exists(): os.symlink(ds, ls, target_is_directory=True)
    print(f'outputs/{sub} -> {ds}')

outputs/checkpoints -> /content/drive/MyDrive/cattle_reid/outputs/checkpoints
outputs/logs -> /content/drive/MyDrive/cattle_reid/outputs/logs
outputs/results -> /content/drive/MyDrive/cattle_reid/outputs/results


## 6. Datos: CMPD300 (source, entrenamiento) + Zenodo (target, eval)

In [6]:
import zipfile
IMG_EXTS = {'.jpg','.jpeg','.png','.bmp','.webp'}

# --- CMPD300 (source) ---
CMPD_ZIP = DRIVE_ROOT / 'CMPD300_Baseline.zip'
CMPD_LOCAL = Path('/content/cmpd300')
if not CMPD_LOCAL.exists():
    CMPD_LOCAL.mkdir(parents=True)
    !unzip -q "{CMPD_ZIP}" -d "{CMPD_LOCAL}"
cand = [CMPD_LOCAL, CMPD_LOCAL/'Baseline'] + [p for p in CMPD_LOCAL.rglob('*') if p.is_dir()]
CMPD_DIR = next((p for p in cand if (p/'train').is_dir()), None)
assert CMPD_DIR, 'No encuentro train/ en CMPD300.'
SOURCE_TRAIN = CMPD_DIR / 'train'
print('SOURCE_TRAIN:', SOURCE_TRAIN, '| ids:', len([d for d in SOURCE_TRAIN.iterdir() if d.is_dir()]))

# --- Zenodo (target) ---
MUZZLE_ZIP = DRIVE_ROOT / 'BeefCattle_Muzzle_Individualized.zip'
MUZZLE_LOCAL = Path('/content/muzzle')
if not MUZZLE_LOCAL.exists():
    MUZZLE_LOCAL.mkdir(parents=True)
    !unzip -q "{MUZZLE_ZIP}" -d "{MUZZLE_LOCAL}"
def find_root(base):
    for p in [base, *[d for d in base.rglob('*') if d.is_dir()]]:
        subs = [d for d in p.iterdir() if d.is_dir()]
        if len(subs) >= 100 and any(d.name.lower().startswith('cattle') for d in subs): return p
    raise RuntimeError('No encuentro cattle_*')
TARGET_DIR = find_root(MUZZLE_LOCAL)
print('TARGET_DIR:', TARGET_DIR, '| ids:', len([d for d in TARGET_DIR.iterdir() if d.is_dir()]))

SOURCE_TRAIN: /content/cmpd300/MuzzleSplit/train | ids: 300
TARGET_DIR: /content/muzzle/BeefCattle_Muzzle_Individualized | ids: 268


## 7. Smoke (~2 min): valida ambos configs antes de la corrida larga

Confirma que ResNet-heavy y DINOv2-heavy entrenan y guardan. El de DINOv2 es el más frágil
(fine-tune de ViT); si falla acá, lo ves en 1 minuto en vez de a la mañana.

In [ ]:
for bk in ['resnet50', 'dinov2']:
    print('==== smoke', bk, '(heavy) ====')
    !python scripts/13_train_loss.py --train-dir "{SOURCE_TRAIN}" --loss supcon --backbone {bk} --aug heavy --smoke --num-workers 2 --out /tmp/smoke_{bk}.pt 2>&1 | tail -3

## 8. Entrenar los dos encoders (resiliente, ~1.5–2 h)

Si un experimento falla, se loguea y sigue con el resto. Reanudable: saltea los checkpoints ya
hechos. Cada log se escribe a Drive (`outputs/logs/train_<tag>.log`) además de mostrarse en vivo.

Si DINOv2 tira `CUDA out of memory`, agregá `'--P','12'` al comando (batch 48).

In [7]:
import subprocess

EXPERIMENTS = [
    # tag, backbone, aug, epochs
    ('supcon_heavyaug',        'resnet50', 'heavy',  100),
    ('supcon_dinov2_heavyaug', 'dinov2',   'heavy',  60),
    ('supcon_dinov2_strong',   'dinov2',   'strong', 60),   # ablación: aisla el efecto del backbone
]
LOGDIR = Path('outputs/logs'); LOGDIR.mkdir(parents=True, exist_ok=True)

done, failed = [], []
for tag, backbone, aug, epochs in EXPERIMENTS:
    out = Path('outputs/checkpoints') / f'cmpd300_{tag}.pt'
    if out.exists():
        print('✓ ya existe, salteo:', out.name); done.append(tag); continue
    logf = LOGDIR / f'train_{tag}.log'
    print(f'\n==== {tag} (backbone={backbone}, aug={aug}, {epochs} ep) -> {logf} ====', flush=True)
    cmd = ['python', '-u', 'scripts/13_train_loss.py', '--train-dir', str(SOURCE_TRAIN),
           '--loss', 'supcon', '--backbone', backbone, '--aug', aug,
           '--epochs', str(epochs), '--num-workers', '2', '--out', str(out)]
    with open(logf, 'w') as fh:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end=''); fh.write(line); fh.flush()
        p.wait()
    if p.returncode == 0 and out.exists():
        print(f'   OK {tag} guardado'); done.append(tag)
    else:
        print(f'   FALLO {tag} (rc={p.returncode}) -- ver {logf}'); failed.append(tag)
print('\nRESUMEN train:  ok =', done, ' | fallaron =', failed)

✓ ya existe, salteo: cmpd300_supcon_heavyaug.pt
✓ ya existe, salteo: cmpd300_supcon_dinov2_heavyaug.pt

==== supcon_dinov2_strong (backbone=dinov2, aug=strong, 60 ep) -> outputs/logs/train_supcon_dinov2_strong.log ====
[13:35:24] INFO loss=supcon | backbone=dinov2 | aug=strong | lr=3.0e-05 | device=cuda | ids=300 | imgs=1500 | P=16 K=4 | epochs=60 | image_size=224 | norm=True
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
[13:36:52] INFO ep 01/60 | loss 2.4343 | lr 8.40e-06
[13:37:57] INFO ep 02/60 | loss 1.6833 | lr 1.38e-05
[13:39:00] INFO ep 03/60 | loss 1.5405 | lr 1.92e-05
[13:40:04] INFO ep 04/60 | loss 1.4651 | lr 2.46e-05
[13:41:08] INFO ep 05/60 | loss 1.3855 | lr 3.00e-05
[13:42:11] INFO ep 06/60 | loss 1.3513 | lr 3.00e-05
[13:43:15] INFO ep 0

## 9. Evaluar todo por el mismo pipeline (pHash -> HDBSCAN)

Compara los dos nuevos contra SupCon original (0.542), ImageNet y DINOv2 congelado.
`ARI@eps0` es el número honesto (params fijos). `best-eps` es diagnóstico ORÁCULO (mira labels).

In [9]:
import numpy as np, json, random
from sklearn.cluster import HDBSCAN
from src.reid.encoders import resnet50_checkpoint, dinov2_checkpoint, build_encoder
from src.reid.reid_dataset import entries_from_folders
from src.reid.phash import dedup_entries
from src.reid.cluster import clustering_metrics, kmeans_reference
from src.reid.eval_reid import rank_metrics

SEED = 0
entries, id_map = entries_from_folders(TARGET_DIR)
entries, dinfo = dedup_entries(entries, TARGET_DIR, threshold=6)
lab = np.array([e['label'] for e in entries]); n_true = len(np.unique(lab))
print(f'eval: {len(entries)} imgs | {n_true} ids | dedup={dinfo}')

def single_shot_idx(labels, seed=0):
    rng = random.Random(seed); by = {}
    for i, l in enumerate(labels.tolist()): by.setdefault(l, []).append(i)
    gal, prb = [], []
    for _l, idxs in sorted(by.items()):
        if len(idxs) < 2: continue
        idxs = idxs[:]; rng.shuffle(idxs); gal.append(idxs[0]); prb += idxs[1:]
    return gal, prb
gal, prb = single_shot_idx(lab, SEED)

MODELS = [
    ('imagenet',                None),
    ('dinov2 (frozen)',         None),
    ('supcon (0.542 original)', 'outputs/checkpoints/cmpd300_supcon.pt'),
    ('supcon_heavyaug',         'outputs/checkpoints/cmpd300_supcon_heavyaug.pt'),
    ('supcon_dinov2_heavyaug',  'outputs/checkpoints/cmpd300_supcon_dinov2_heavyaug.pt'),
    ('supcon_dinov2_strong',    'outputs/checkpoints/cmpd300_supcon_dinov2_strong.pt'),
]
def load(name, ckpt):
    if ckpt is None: return build_encoder('dinov2' if 'dinov2' in name else 'imagenet')
    return dinov2_checkpoint(ckpt) if 'dinov2' in name else resnet50_checkpoint(ckpt)

results = {}
print(f"\n{'modelo':26} {'ARI@eps0':>9} {'best-eps(oraculo)':>18} {'kmeans':>7} {'Rank-1':>7}")
print('-' * 78)
for name, ckpt in MODELS:
    if ckpt and not Path(ckpt).is_file():
        print(f'{name:26}  (falta checkpoint)'); continue
    try:
        emb, lb = load(name, ckpt).embed(entries, data_dir=TARGET_DIR, batch_size=64, num_workers=2)
        base = clustering_metrics(lb, HDBSCAN(min_cluster_size=2, metric='cosine').fit_predict(emb))
        best_ari, best_eps = base['ARI'], 0.0
        for e in [0.02, 0.022, 0.025, 0.027, 0.03, 0.033]:
            m = clustering_metrics(lb, HDBSCAN(min_cluster_size=2, metric='cosine',
                                   cluster_selection_epsilon=e).fit_predict(emb))
            if m['ARI'] > best_ari: best_ari, best_eps = m['ARI'], e
        km = clustering_metrics(lb, kmeans_reference(emb, k=n_true, seed=SEED))['ARI']
        rk = rank_metrics(emb[prb], lb[prb], emb[gal], lb[gal])
        results[name] = {'ari_eps0': base['ARI'], 'nmi_eps0': base['NMI'],
                         'best_ari_oracle': best_ari, 'best_eps_oracle': best_eps,
                         'kmeans_ari': km, 'rank1': round(rk['rank1'], 4)}
        print(f"{name:26} {base['ARI']:>9.3f} {f'{best_ari:.3f}@{best_eps}':>18} {km:>7.3f} {rk['rank1']:>7.3f}")
    except Exception as ex:
        print(f'{name:26}  ERROR: {ex}'); results[name] = {'error': str(ex)}

Path('outputs/results/16_night_comparison.json').write_text(
    json.dumps({'n_true': n_true, 'n_eval': len(entries), 'models': results}, indent=2))
print(f'\n(#ids reales={n_true}) -> outputs/results/16_night_comparison.json')
print('ARI@eps0 = numero HONESTO. best-eps = diagnostico ORACULO (mira labels, NO reportable).')

eval: 1554 imgs | 268 ids | dedup={'n_input': 4923, 'n_kept': 1554, 'n_dropped': 3369, 'threshold': 6}

modelo                      ARI@eps0  best-eps(oraculo)  kmeans  Rank-1
------------------------------------------------------------------------------


[14:49:19] INFO encoder='imagenet_resnet50' embedded 1554 imgs → dim 2048
INFO:reid.encoders:encoder='imagenet_resnet50' embedded 1554 imgs → dim 2048


imagenet                       0.461        0.478@0.033   0.737   0.803


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
[14:50:41] INFO encoder='dinov2-base' embedded 1554 imgs → dim 768
INFO:reid.encoders:encoder='dinov2-base' embedded 1554 imgs → dim 768


dinov2 (frozen)                0.150          0.150@0.0   0.574   0.667


...

supcon_heavyaug                0.483         0.635@0.02   0.717   0.778


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
[14:53:47] INFO encoder='cmpd300_supcon_dinov2_heavyaug.pt' embedded 1554 imgs → dim 768
INFO:reid.encoders:encoder='cmpd300_supcon_dinov2_heavyaug.pt' embedded 1554 imgs → dim 768


supcon_dinov2_heavyaug         0.663        0.760@0.033   0.779   0.865


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
[14:54:42] INFO encoder='cmpd300_supcon_dinov2_strong.pt' embedded 1554 imgs → dim 768
INFO:reid.encoders:encoder='cmpd300_supcon_dinov2_strong.pt' embedded 1554 imgs → dim 768


supcon_dinov2_strong           0.687        0.773@0.033   0.788   0.876

(#ids reales=268) -> outputs/results/16_night_comparison.json
ARI@eps0 = numero HONESTO. best-eps = diagnostico ORACULO (mira labels, NO reportable).


## 10. Verificación final: ¿quedó todo en Drive?

In [ ]:
CKPT = DRIVE_ROOT / 'outputs' / 'checkpoints'
RES  = DRIVE_ROOT / 'outputs' / 'results'
LOG  = DRIVE_ROOT / 'outputs' / 'logs'
ok = True
print('Checkpoints en Drive:')
for tag in ['supcon_heavyaug', 'supcon_dinov2_heavyaug']:
    f = CKPT / f'cmpd300_{tag}.pt'
    size = f'  ({f.stat().st_size/1e6:.0f} MB)' if f.exists() else ''
    print(f'   {"OK " if f.exists() else "FALTA"}  {f.name}{size}'); ok = ok and f.exists()
rj = RES / '16_night_comparison.json'
print('\nResultados:', 'OK' if rj.exists() else 'FALTA', rj.name)
print('Logs:', [p.name for p in sorted(LOG.glob('train_supcon_*.log'))])
print('\n' + ('TODO GUARDADO -- podes cerrar Colab.' if (ok and rj.exists())
              else 'FALTA algo arriba -- revisa los logs.'))

## Cómo leer los resultados

- **`ARI@eps0`** es el número honesto de cada encoder (HDBSCAN, params fijos). Compará
  `supcon_heavyaug` y `supcon_dinov2_heavyaug` contra el `0.542` original, ImageNet y DINOv2.
- **`best-eps(oráculo)`** es el techo si tuvieras el eps ideal — diagnóstico, no reportable
  (elige mirando labels del target).
- Si `supcon_heavyaug` **baja** en vez de subir, la augmentation fue demasiado agresiva y
  destruyó señal del hocico → hay que aflojarla.
- DINOv2 fine-tune es el experimento más frágil; si falló, el log `train_supcon_dinov2_heavyaug.log`
  te dice por qué (OOM, o loss que no baja = LR).

In [10]:
import numpy as np, json, random
from sklearn.cluster import HDBSCAN
from src.reid.encoders import resnet50_checkpoint, dinov2_checkpoint, build_encoder
from src.reid.reid_dataset import entries_from_folders
from src.reid.phash import dedup_entries
from src.reid.cluster import clustering_metrics, kmeans_reference
from src.reid.eval_reid import rank_metrics

SEED = 0
entries, id_map = entries_from_folders(TARGET_DIR)
entries, dinfo = dedup_entries(entries, TARGET_DIR, threshold=6)
lab = np.array([e['label'] for e in entries]); n_true = len(np.unique(lab))
print(f'eval: {len(entries)} imgs | {n_true} ids | dedup={dinfo}')

def single_shot_idx(labels, seed=0):
    rng = random.Random(seed); by = {}
    for i, l in enumerate(labels.tolist()): by.setdefault(l, []).append(i)
    gal, prb = [], []
    for _l, idxs in sorted(by.items()):
        if len(idxs) < 2: continue
        idxs = idxs[:]; rng.shuffle(idxs); gal.append(idxs[0]); prb += idxs[1:]
    return gal, prb
gal, prb = single_shot_idx(lab, SEED)

MODELS = [
    ('supcon_dinov2_heavyaug',  'outputs/checkpoints/cmpd300_supcon_dinov2_heavyaug.pt'),
    ('supcon_dinov2_strong',    'outputs/checkpoints/cmpd300_supcon_dinov2_strong.pt')
]
def load(name, ckpt):
    if ckpt is None: return build_encoder('dinov2' if 'dinov2' in name else 'imagenet')
    return dinov2_checkpoint(ckpt) if 'dinov2' in name else resnet50_checkpoint(ckpt)

results = {}
print(f"\n{'modelo':26} {'ARI@eps0':>9} {'best-eps(oraculo)':>18} {'kmeans':>7} {'Rank-1':>7}")
print('-' * 78)
for name, ckpt in MODELS:
    if ckpt and not Path(ckpt).is_file():
        print(f'{name:26}  (falta checkpoint)'); continue
    try:
        emb, lb = load(name, ckpt).embed(entries, data_dir=TARGET_DIR, batch_size=64, num_workers=2)
        base = clustering_metrics(lb, HDBSCAN(min_cluster_size=2, metric='cosine').fit_predict(emb))
        best_ari, best_eps = base['ARI'], 0.0
        for e in [0.033, 0.035, 0.037, 0.04, 0.031]:
            m = clustering_metrics(lb, HDBSCAN(min_cluster_size=2, metric='cosine',
                                   cluster_selection_epsilon=e).fit_predict(emb))
            if m['ARI'] > best_ari: best_ari, best_eps = m['ARI'], e
        km = clustering_metrics(lb, kmeans_reference(emb, k=n_true, seed=SEED))['ARI']
        rk = rank_metrics(emb[prb], lb[prb], emb[gal], lb[gal])
        results[name] = {'ari_eps0': base['ARI'], 'nmi_eps0': base['NMI'],
                         'best_ari_oracle': best_ari, 'best_eps_oracle': best_eps,
                         'kmeans_ari': km, 'rank1': round(rk['rank1'], 4)}
        print(f"{name:26} {base['ARI']:>9.3f} {f'{best_ari:.3f}@{best_eps}':>18} {km:>7.3f} {rk['rank1']:>7.3f}")
    except Exception as ex:
        print(f'{name:26}  ERROR: {ex}'); results[name] = {'error': str(ex)}

Path('outputs/results/16_night_comparison.json').write_text(
    json.dumps({'n_true': n_true, 'n_eval': len(entries), 'models': results}, indent=2))
print(f'\n(#ids reales={n_true}) -> outputs/results/16_night_comparison.json')
print('ARI@eps0 = numero HONESTO. best-eps = diagnostico ORACULO (mira labels, NO reportable).')

eval: 1554 imgs | 268 ids | dedup={'n_input': 4923, 'n_kept': 1554, 'n_dropped': 3369, 'threshold': 6}

modelo                      ARI@eps0  best-eps(oraculo)  kmeans  Rank-1
------------------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
[14:59:31] INFO encoder='cmpd300_supcon_dinov2_heavyaug.pt' embedded 1554 imgs → dim 768
INFO:reid.encoders:encoder='cmpd300_supcon_dinov2_heavyaug.pt' embedded 1554 imgs → dim 768


supcon_dinov2_heavyaug         0.663         0.773@0.04   0.779   0.865


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
[15:00:26] INFO encoder='cmpd300_supcon_dinov2_strong.pt' embedded 1554 imgs → dim 768
INFO:reid.encoders:encoder='cmpd300_supcon_dinov2_strong.pt' embedded 1554 imgs → dim 768


supcon_dinov2_strong           0.687        0.799@0.037   0.788   0.876

(#ids reales=268) -> outputs/results/16_night_comparison.json
ARI@eps0 = numero HONESTO. best-eps = diagnostico ORACULO (mira labels, NO reportable).
